In [0]:
display(spark.sql("SELECT * FROM ecommerce_dev.silver.products LIMIT 5"))

product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,bronze_ingested_at,has_missing_dimensions
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-08-04T00:55:54.392Z,false
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-08-04T00:55:54.392Z,false
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-08-04T00:55:54.392Z,false
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-08-04T00:55:54.392Z,false
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-08-04T00:55:54.392Z,false


In [0]:
display(spark.sql("SELECT * FROM ecommerce_dev.silver.category_translation LIMIT 5"))

product_category_name,product_category_name_english,bronze_ingested_at,is_backfilled
beleza_saude,health_beauty,2026-08-04T00:56:12.497Z,false
informatica_acessorios,computers_accessories,2026-08-04T00:56:12.497Z,false
automotivo,auto,2026-08-04T00:56:12.497Z,false
cama_mesa_banho,bed_bath_table,2026-08-04T00:56:12.497Z,false
moveis_decoracao,furniture_decor,2026-08-04T00:56:12.497Z,false


### Step 1: Create the target table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.dim_product (
    product_key BIGINT GENERATED ALWAYS AS IDENTITY,
    product_id STRING NOT NULL,
    product_category_name STRING,
    product_category_name_english STRING,
    product_name_length INT,
    product_description_length INT,
    product_photos_qty INT,
    product_weight_g INT,
    product_length_cm INT,
    product_height_cm INT,
    product_width_cm INT,
    has_missing_dimensions BOOLEAN,
    category_is_backfilled BOOLEAN,
    effective_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN,
    gold_updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Gold product dimension - SCD Type 2. Tracks history of product attribute and category changes. Grain: one row per product_id per version.';

### Step 2: Stage the source (join products to category_translation)

In [0]:
from pyspark.sql.functions import *

stg_products = spark.sql("""
    SELECT
        p.product_id,
        p.product_category_name,
        c.product_category_name_english,
        p.product_name_length,
        p.product_description_length,
        p.product_photos_qty,
        p.product_weight_g,
        p.product_length_cm,
        p.product_height_cm,
        p.product_width_cm,
        p.has_missing_dimensions,
        coalesce(c.is_backfilled, false) AS category_is_backfilled
    FROM ecommerce_dev.silver.products p
    LEFT JOIN ecommerce_dev.silver.category_translation c
        ON p.product_category_name = c.product_category_name
""")

stg_products.createOrReplaceTempView("stg_products")

# Sanity check: any products with no category match at all (shouldn't happen given your 3 backfilled entries, but confirm)
display(stg_products.filter(col("product_category_name_english").isNull()))

product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_missing_dimensions,category_is_backfilled


### Step 3: Close out changed current rows

In [0]:
spark.sql("""
    MERGE INTO ecommerce_dev.gold.dim_product AS target
    USING stg_products AS source
    ON target.product_id = source.product_id AND target.is_current = true

    WHEN MATCHED AND (
        target.product_category_name <> source.product_category_name OR
        target.product_category_name_english <> source.product_category_name_english OR
        target.product_name_length <> source.product_name_length OR
        target.product_description_length <> source.product_description_length OR
        target.product_photos_qty <> source.product_photos_qty OR
        target.product_weight_g <> source.product_weight_g OR
        target.product_length_cm <> source.product_length_cm OR
        target.product_height_cm <> source.product_height_cm OR
        target.product_width_cm <> source.product_width_cm OR
        target.has_missing_dimensions <> source.has_missing_dimensions OR
        target.category_is_backfilled <> source.category_is_backfilled
    ) THEN UPDATE SET
        target.end_date = current_timestamp(),
        target.is_current = false,
        target.gold_updated_at = current_timestamp()
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

Step 4: Insert new products + new versions of changed products

In [0]:
spark.sql("""
    MERGE INTO ecommerce_dev.gold.dim_product AS target
    USING stg_products AS source
    ON target.product_id = source.product_id AND target.is_current = true

    WHEN NOT MATCHED THEN INSERT (
        product_id, product_category_name, product_category_name_english,
        product_name_length, product_description_length, product_photos_qty,
        product_weight_g, product_length_cm, product_height_cm, product_width_cm,
        has_missing_dimensions, category_is_backfilled,
        effective_date, end_date, is_current, gold_updated_at
    ) VALUES (
        source.product_id, source.product_category_name, source.product_category_name_english,
        source.product_name_length, source.product_description_length, source.product_photos_qty,
        source.product_weight_g, source.product_length_cm, source.product_height_cm, source.product_width_cm,
        source.has_missing_dimensions, source.category_is_backfilled,
        current_timestamp(), NULL, true, current_timestamp()
    )
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Step 5: NOT NULL + PK, comment, verify

In [0]:
%sql
ALTER TABLE ecommerce_dev.gold.dim_product ALTER COLUMN product_key SET NOT NULL;

ALTER TABLE ecommerce_dev.gold.dim_product 
ADD CONSTRAINT pk_dim_product PRIMARY KEY (product_key);

COMMENT ON TABLE ecommerce_dev.gold.dim_product IS 
'Gold product dimension - SCD Type 2. Tracks history of product attributes and category. Grain: one row per product_id per version. Surrogate key: product_key. is_current flags the active version.';

In [0]:
silver_ct = spark.table("ecommerce_dev.silver.products").count()
gold_current_ct = spark.sql("SELECT count(*) as ct FROM ecommerce_dev.gold.dim_product WHERE is_current = true").collect()[0]['ct']
gold_total_ct = spark.table("ecommerce_dev.gold.dim_product").count()

print(f"Silver: {silver_ct} | Gold current: {gold_current_ct} | Gold total: {gold_total_ct}")
print(f"Current matches Silver: {silver_ct == gold_current_ct}")

display(spark.sql("SELECT * FROM ecommerce_dev.gold.dim_product LIMIT 5"))

Silver: 32951 | Gold current: 32951 | Gold total: 32951
Current matches Silver: True


product_key,product_id,product_category_name,product_category_name_english,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,has_missing_dimensions,category_is_backfilled,effective_date,end_date,is_current,gold_updated_at
1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery,40,287,1,225,16,10,14,false,false,2026-08-09T17:45:10.050Z,null,true,2026-08-09T17:45:10.050Z
2,3aa071139cb16b67ca9e5dea641aaa2f,artes,art,44,276,1,1000,30,18,20,false,false,2026-08-09T17:45:10.050Z,null,true,2026-08-09T17:45:10.050Z
3,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure,46,250,1,154,18,9,15,false,false,2026-08-09T17:45:10.050Z,null,true,2026-08-09T17:45:10.050Z
4,cef67bcfe19066a932b7673e239eb23d,bebes,baby,27,261,1,371,26,4,26,false,false,2026-08-09T17:45:10.050Z,null,true,2026-08-09T17:45:10.050Z
5,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares,37,402,4,625,20,17,13,false,false,2026-08-09T17:45:10.050Z,null,true,2026-08-09T17:45:10.050Z
